# 03 — Legal Document Chunking

This notebook implements the third stage of the **CLARA** document ingestion pipeline.

It transforms the article-level legal structures generated by `02_document_parsing.ipynb` into retrieval-oriented chunks suitable for semantic embedding and vector indexing.

The chunking strategy preserves the legal provenance of every text segment, including the source document, article, article title, page range, and position within the article. This metadata is essential for traceability and for presenting meaningful legal sources after retrieval.

The main output follows the structure:

```text
data/processed/<DOCUMENT_ID>/chunks.json
```

## 1. Imports

The required Python libraries are loaded for filesystem operations, JSON processing, regular-expression based text handling, and tabular validation of the generated chunks.

In [1]:
from pathlib import Path
import json

import pandas as pd


## 2. Project Paths

The notebook operates on the parsed legal documents stored under `data/processed/`.

Paths are defined relative to the project root so that the chunking pipeline remains portable and does not depend on machine-specific absolute locations.

In [2]:
PROJECT_ROOT = Path("..").resolve()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(PROCESSED_DIR)


C:\Users\user\Documents\GitHub\clara-rag-pt\data\processed


## 2. Project Paths

The notebook operates on the parsed legal documents stored under `data/processed/`.

Paths are defined relative to the project root so that the chunking pipeline remains portable and does not depend on machine-specific absolute locations.

In [3]:
parsed_files = sorted(
    PROCESSED_DIR.glob("*/parsed.json")
)

print(f"Foram encontrados {len(parsed_files)} documentos:\n")

for path in parsed_files:
    print(f"- {path.parent.name}")


Foram encontrados 10 documentos:

- CIMI
- CIRC
- CIRS
- CIS
- CIVA
- EBF
- LGT
- OE2026
- RGIT
- RITI


### Processing Step 4

This block performs an additional transformation or validation step in the legal-document chunking workflow.

Its output contributes to ensuring that the generated retrieval units are structurally consistent and retain the metadata required by the subsequent embedding and indexing stages.

In [4]:
MAX_WORDS = 350
OVERLAP_WORDS = 50

print(f"Máximo de palavras por chunk: {MAX_WORDS}")
print(f"Overlap: {OVERLAP_WORDS}")


Máximo de palavras por chunk: 350
Overlap: 50


## 4. Text Segmentation Utilities

This block defines helper logic used to divide long legal text into smaller textual units while attempting to preserve meaningful boundaries.

Legal provisions should not be split arbitrarily whenever avoidable. These utilities provide the lower-level segmentation behaviour used by the article chunking strategy.

In [5]:
def count_words(text: str) -> int:
    """Conta aproximadamente o número de palavras de um texto."""
    return len(text.split())


### Processing Step 6

This block performs an additional transformation or validation step in the legal-document chunking workflow.

Its output contributes to ensuring that the generated retrieval units are structurally consistent and retain the metadata required by the subsequent embedding and indexing stages.

In [6]:
def build_context_header(
    document_id: str,
    article: dict
) -> str:
    """Constrói um cabeçalho textual com contexto jurídico do artigo."""

    parts = [document_id]

    hierarchy = article.get("hierarchy", {})

    for level in [
        "part",
        "book",
        "title",
        "chapter",
        "section",
        "subsection"
    ]:
        value = hierarchy.get(level)

        if value:
            parts.append(value)

    parts.append(article["article"])

    article_title = article.get("article_title")

    if article_title:
        parts.append(article_title)

    return "\n".join(parts)


## 4. Text Segmentation Utilities

This block defines helper logic used to divide long legal text into smaller textual units while attempting to preserve meaningful boundaries.

Legal provisions should not be split arbitrarily whenever avoidable. These utilities provide the lower-level segmentation behaviour used by the article chunking strategy.

In [7]:
def split_paragraphs(text: str) -> list[str]:
    """Divide o texto em parágrafos/linhas relevantes."""

    paragraphs = []

    for paragraph in text.splitlines():

        paragraph = paragraph.strip()

        if paragraph:
            paragraphs.append(paragraph)

    return paragraphs


## 5. Article-Aware Chunk Generation

This block defines the core chunking logic.

Each parsed legal article is converted into one or more retrieval units. Short articles can remain as a single chunk, while longer provisions are divided into smaller segments according to the configured size constraints.

Every generated chunk retains its legal metadata so that semantic retrieval results can always be traced back to their source article and document.

In [8]:
def chunk_text(
    text: str,
    max_words: int = 350,
    overlap_words: int = 50
) -> list[str]:
    """
    Divide o texto preservando, tanto quanto possível,
    os parágrafos originais.
    """

    paragraphs = split_paragraphs(text)

    if not paragraphs:
        return []

    chunks = []
    current_words = []

    for paragraph in paragraphs:

        paragraph_words = paragraph.split()

        # Um único parágrafo ultrapassa o limite.
        if len(paragraph_words) > max_words:

            if current_words:
                chunks.append(" ".join(current_words))
                current_words = []

            start = 0

            while start < len(paragraph_words):

                end = min(
                    start + max_words,
                    len(paragraph_words)
                )

                chunk_words = paragraph_words[start:end]

                chunks.append(
                    " ".join(chunk_words)
                )

                if end >= len(paragraph_words):
                    break

                start = max(
                    end - overlap_words,
                    start + 1
                )

            continue

        # Fechar o chunk atual antes de ultrapassar o limite.
        if (
            current_words
            and len(current_words) + len(paragraph_words) > max_words
        ):

            chunks.append(
                " ".join(current_words)
            )

            overlap = (
                current_words[-overlap_words:]
                if overlap_words > 0
                else []
            )

            current_words = overlap.copy()

        current_words.extend(paragraph_words)

    if current_words:
        chunks.append(
            " ".join(current_words)
        )

    return chunks


## 2. Project Paths

The notebook operates on the parsed legal documents stored under `data/processed/`.

Paths are defined relative to the project root so that the chunking pipeline remains portable and does not depend on machine-specific absolute locations.

In [9]:
# Teste num documento real

test_path = (
    PROCESSED_DIR
    / "CIRS"
    / "parsed.json"
)

if not test_path.exists():
    test_path = parsed_files[0] if parsed_files else None

if test_path is None:
    raise FileNotFoundError(
        "Não foram encontrados ficheiros parsed.json."
    )

with open(
    test_path,
    "r",
    encoding="utf-8"
) as f:
    test_document = json.load(f)

test_article = test_document["articles"][0]

print(f"Documento: {test_document['document_id']}")
print(f"Artigo: {test_article['article']}")
print()
print(
    build_context_header(
        test_document["document_id"],
        test_article
    )
)


Documento: CIRS
Artigo: Artigo 1.º

CIRS
Artigo 1.º
Aprovação do Código


## 6. Chunking Test

Before processing the complete corpus, the chunking procedure is applied to a sample document or article.

This intermediate test provides a convenient point for inspecting the resulting chunk structure and verifying that legal metadata and textual content are preserved correctly.

In [10]:
article_text = test_article["text"]

test_chunks = chunk_text(
    article_text,
    max_words=MAX_WORDS,
    overlap_words=OVERLAP_WORDS
)

print(f"Palavras no artigo: {count_words(article_text)}")
print(f"Chunks gerados: {len(test_chunks)}")

for i, chunk in enumerate(test_chunks, start=1):
    print(
        f"Chunk {i}: {count_words(chunk)} palavras"
    )


Palavras no artigo: 19
Chunks gerados: 1
Chunk 1: 19 palavras


## 5. Article-Aware Chunk Generation

This block defines the core chunking logic.

Each parsed legal article is converted into one or more retrieval units. Short articles can remain as a single chunk, while longer provisions are divided into smaller segments according to the configured size constraints.

Every generated chunk retains its legal metadata so that semantic retrieval results can always be traced back to their source article and document.

In [11]:
def create_article_chunks(
    document_id: str,
    article: dict,
    article_index: int,
    max_words: int = 350,
    overlap_words: int = 50
) -> list[dict]:
    """
    Cria um ou mais chunks para um artigo.

    O article_index é um identificador técnico sequencial
    e garante unicidade mesmo quando o mesmo número de artigo
    surge mais de uma vez no documento.
    """

    article_text = article.get("text", "").strip()

    if not article_text:
        return []

    text_chunks = chunk_text(
        article_text,
        max_words=max_words,
        overlap_words=overlap_words
    )

    context_header = build_context_header(
        document_id,
        article
    )

    chunks = []
    total_chunks = len(text_chunks)

    for chunk_index, chunk_text_value in enumerate(
        text_chunks,
        start=1
    ):

        chunk_id = (
            f"{document_id}_"
            f"ART_{article_index:04d}_"
            f"CHUNK_{chunk_index:03d}"
        )

        embedding_text = (
            f"{context_header}\n\n"
            f"{chunk_text_value}"
        )

        chunks.append({
            "chunk_id": chunk_id,
            "document_id": document_id,
            "article_index": article_index,
            "article": article["article"],
            "article_number": article["article_number"],
            "article_title": article.get("article_title"),
            "page_start": article["page_start"],
            "page_end": article["page_end"],
            "hierarchy": article.get("hierarchy", {}),
            "chunk_index": chunk_index,
            "total_chunks": total_chunks,
            "word_count": count_words(chunk_text_value),
            "text": chunk_text_value,
            "embedding_text": embedding_text
        })

    return chunks


## 5. Article-Aware Chunk Generation

This block defines the core chunking logic.

Each parsed legal article is converted into one or more retrieval units. Short articles can remain as a single chunk, while longer provisions are divided into smaller segments according to the configured size constraints.

Every generated chunk retains its legal metadata so that semantic retrieval results can always be traced back to their source article and document.

In [12]:
def chunk_document(
    parsed_document: dict,
    max_words: int = 350,
    overlap_words: int = 50
) -> dict:
    """Transforma todos os artigos de um documento em chunks."""

    document_id = parsed_document["document_id"]

    all_chunks = []

    for article_index, article in enumerate(
        parsed_document["articles"],
        start=1
    ):

        article_chunks = create_article_chunks(
            document_id=document_id,
            article=article,
            article_index=article_index,
            max_words=max_words,
            overlap_words=overlap_words
        )

        all_chunks.extend(article_chunks)

    return {
        "document_id": document_id,
        "source_file": parsed_document["source_file"],
        "num_articles": parsed_document["num_articles"],
        "num_chunks": len(all_chunks),
        "chunking": {
            "max_words": max_words,
            "overlap_words": overlap_words
        },
        "chunks": all_chunks
    }


## 6. Chunking Test

Before processing the complete corpus, the chunking procedure is applied to a sample document or article.

This intermediate test provides a convenient point for inspecting the resulting chunk structure and verifying that legal metadata and textual content are preserved correctly.

In [13]:
chunked_document = chunk_document(
    test_document,
    max_words=MAX_WORDS,
    overlap_words=OVERLAP_WORDS
)

print(f"Documento: {chunked_document['document_id']}")
print(f"Artigos: {chunked_document['num_articles']}")
print(f"Chunks: {chunked_document['num_chunks']}")


Documento: CIRS
Artigos: 216
Chunks: 370


### 5.1 Stable Chunk Identification and Metadata

Each retrieval unit is associated with a deterministic chunk identifier and metadata describing its legal origin.

The identifier distinguishes individual chunks while the metadata preserves information such as the document, article, title, page range, and chunk position. Unique identifiers are particularly important when vectors are later uploaded to Qdrant.

In [14]:
df_chunks = pd.DataFrame(
    chunked_document["chunks"]
)

columns_to_show = [
    "chunk_id",
    "article",
    "article_title",
    "chunk_index",
    "total_chunks",
    "word_count",
    "page_start",
    "page_end"
]

df_chunks[columns_to_show].head(20)


,chunk_id,article,article_title,chunk_index,total_chunks,word_count,page_start,page_end
0,CIRS_ART_0001_CHUNK_001,Artigo 1.º,Aprovação do Código,1,1,19,7,7
1,CIRS_ART_0002_CHUNK_001,Artigo 2.º,Entrada em vigor,1,1,13,7,7
2,CIRS_ART_0003_CHUNK_001,Artigo 3.º,Impostos abolidos,1,1,98,7,7
3,CIRS_ART_0004_CHUNK_001,Artigo 3.º-A,Regime transitório de enquadramento dos agente...,1,2,341,7,8
4,CIRS_ART_0004_CHUNK_002,Artigo 3.º-A,Regime transitório de enquadramento dos agente...,2,2,88,7,8
5,CIRS_ART_0005_CHUNK_001,Artigo 4.º,Regime transitório da categoria B,1,1,214,8,8
6,CIRS_ART_0006_CHUNK_001,Artigo 5.º,Regime transitório da categoria G,1,1,213,8,9
7,CIRS_ART_0007_CHUNK_001,Artigo 6.º,Reporte de rendimentos,1,1,24,9,9
8,CIRS_ART_0008_CHUNK_001,Artigo 7.º,Obrigação de contabilidade organizada,1,1,63,9,9
9,CIRS_ART_0009_CHUNK_001,Artigo 8.º,Modelos de impressos,1,1,21,9,9


### Processing Step 15

This block performs an additional transformation or validation step in the legal-document chunking workflow.

Its output contributes to ensuring that the generated retrieval units are structurally consistent and retain the metadata required by the subsequent embedding and indexing stages.

In [15]:
df_chunks["word_count"].describe()


count    370.000000
mean     218.459459
std      130.393047
min        1.000000
25%       87.000000
50%      257.000000
75%      342.000000
max      350.000000
Name: word_count, dtype: float64

### 7.1 Chunk Identifier Uniqueness

Chunk identifiers must be unique across the complete corpus.

This validation detects duplicate IDs before vector indexing. Duplicate identifiers could otherwise cause collisions or overwrites when the chunks are stored in Qdrant.

In [16]:
split_articles = (
    df_chunks.loc[
        df_chunks["total_chunks"] > 1,
        "article_index"
    ]
    .nunique()
)

print(
    "Artigos divididos em mais de um chunk: "
    f"{split_articles}"
)


Artigos divididos em mais de um chunk: 54


## 10. Corpus-Level Summary

Aggregate statistics are calculated for the chunked corpus.

These values provide a reference for the number of documents, articles, and retrieval chunks produced by the preprocessing pipeline and can later be compared with the number of generated embeddings and indexed Qdrant points.

In [17]:
chunk_distribution = (
    df_chunks
    .groupby("article_index")["total_chunks"]
    .first()
    .value_counts()
    .sort_index()
)

chunk_distribution


total_chunks
1     162
2      25
3       7
4      11
5       4
6       3
7       1
12      1
13      1
23      1
Name: count, dtype: int64

### 5.1 Stable Chunk Identification and Metadata

Each retrieval unit is associated with a deterministic chunk identifier and metadata describing its legal origin.

The identifier distinguishes individual chunks while the metadata preserves information such as the document, article, title, page range, and chunk position. Unique identifiers are particularly important when vectors are later uploaded to Qdrant.

In [18]:
df_chunks.sort_values(
    "word_count",
    ascending=False
)[
    [
        "chunk_id",
        "article",
        "article_title",
        "word_count"
    ]
].head(20)


,chunk_id,article,article_title,word_count
31,CIRS_ART_0017_CHUNK_015,Artigo 16.º,As modificações do Código,350
30,CIRS_ART_0017_CHUNK_014,Artigo 16.º,As modificações do Código,350
17,CIRS_ART_0017_CHUNK_001,Artigo 16.º,As modificações do Código,350
54,CIRS_ART_0024_CHUNK_001,Artigo 5.º,Rendimentos da categoria E,350
18,CIRS_ART_0017_CHUNK_002,Artigo 16.º,As modificações do Código,350
324,CIRS_ART_0177_CHUNK_001,Artigo 116.º,Registos,350
87,CIRS_ART_0033_CHUNK_002,Artigo 12.º-A,Regime fiscal aplicável a ex-residentes,350
43,CIRS_ART_0019_CHUNK_003,Artigo 2.º,Rendimentos da categoria A,350
158,CIRS_ART_0074_CHUNK_002,Artigo 43.º,Mais-valias,350
159,CIRS_ART_0074_CHUNK_003,Artigo 43.º,Mais-valias,350


### 5.1 Stable Chunk Identification and Metadata

Each retrieval unit is associated with a deterministic chunk identifier and metadata describing its legal origin.

The identifier distinguishes individual chunks while the metadata preserves information such as the document, article, title, page range, and chunk position. Unique identifiers are particularly important when vectors are later uploaded to Qdrant.

In [19]:
df_chunks.sort_values(
    "word_count"
)[
    [
        "chunk_id",
        "article",
        "article_title",
        "word_count"
    ]
].head(20)


,chunk_id,article,article_title,word_count
350,CIRS_ART_0197_CHUNK_001,Artigo 134.º,Dever de fiscalização em especial,1
351,CIRS_ART_0198_CHUNK_001,Artigo 135.º,Declarações e outros documentos,1
353,CIRS_ART_0200_CHUNK_001,Artigo 137.º,Garantia de observância de obrigações fiscais,1
352,CIRS_ART_0199_CHUNK_001,Artigo 136.º,Assinatura das declarações,1
348,CIRS_ART_0195_CHUNK_001,Artigo 132.º,Entidades fiscalizadoras,1
349,CIRS_ART_0196_CHUNK_001,Artigo 133.º,Dever de colaboração,1
317,CIRS_ART_0170_CHUNK_001,Artigo 109.º,Compensação,1
261,CIRS_ART_0128_CHUNK_001,Artigo 83.º,Despesas de educação e formação,1
266,CIRS_ART_0132_CHUNK_001,Artigo 85.º-A,Deduções ambientais,1
265,CIRS_ART_0131_CHUNK_001,Artigo 85.º,Encargos com imóveis,1


## 5. Article-Aware Chunk Generation

This block defines the core chunking logic.

Each parsed legal article is converted into one or more retrieval units. Short articles can remain as a single chunk, while longer provisions are divided into smaller segments according to the configured size constraints.

Every generated chunk retains its legal metadata so that semantic retrieval results can always be traced back to their source article and document.

In [20]:
def save_chunked_document(
    chunked_document: dict,
    output_dir: Path
) -> Path:
    """Guarda os chunks de um documento em chunks.json."""

    document_id = chunked_document["document_id"]

    document_dir = (
        output_dir
        / document_id
    )

    document_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_path = (
        document_dir
        / "chunks.json"
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            chunked_document,
            f,
            ensure_ascii=False,
            indent=2
        )

    return output_path


## 2. Project Paths

The notebook operates on the parsed legal documents stored under `data/processed/`.

Paths are defined relative to the project root so that the chunking pipeline remains portable and does not depend on machine-specific absolute locations.

In [21]:
results = []

for parsed_path in parsed_files:

    document_id = parsed_path.parent.name

    print(
        f"A processar {document_id}...",
        end=" "
    )

    try:

        with open(
            parsed_path,
            "r",
            encoding="utf-8"
        ) as f:

            parsed_document = json.load(f)

        chunked_document = chunk_document(
            parsed_document,
            max_words=MAX_WORDS,
            overlap_words=OVERLAP_WORDS
        )

        output_path = save_chunked_document(
            chunked_document,
            PROCESSED_DIR
        )

        results.append({
            "document_id": document_id,
            "articles": chunked_document[
                "num_articles"
            ],
            "chunks": chunked_document[
                "num_chunks"
            ],
            "status": "OK"
        })

        print(
            f"OK "
            f"({chunked_document['num_chunks']} chunks)"
        )

    except Exception as e:

        results.append({
            "document_id": document_id,
            "articles": None,
            "chunks": None,
            "status": f"ERROR: {e}"
        })

        print(f"ERRO: {e}")


A processar CIMI... OK (226 chunks)
A processar CIRC... OK (305 chunks)
A processar CIRS... OK (370 chunks)
A processar CIS... OK (130 chunks)
A processar CIVA... OK (354 chunks)
A processar EBF... OK (279 chunks)
A processar LGT... OK (155 chunks)
A processar OE2026... OK (404 chunks)
A processar RGIT... OK (165 chunks)
A processar RITI... OK (168 chunks)


## 7. Chunk Inspection and Validation

The generated chunks are converted into a tabular representation for inspection.

A DataFrame makes it easier to review chunk identifiers, source documents, article references, page ranges, chunk positions, and text content, and to detect structural problems before embedding generation.

In [22]:
df_results = pd.DataFrame(results)

df_results


,document_id,articles,chunks,status
0,CIMI,192,226,OK
1,CIRC,171,305,OK
2,CIRS,216,370,OK
3,CIS,93,130,OK
4,CIVA,218,354,OK
5,EBF,183,279,OK
6,LGT,124,155,OK
7,OE2026,277,404,OK
8,RGIT,147,165,OK
9,RITI,151,168,OK


## 10. Corpus-Level Summary

Aggregate statistics are calculated for the chunked corpus.

These values provide a reference for the number of documents, articles, and retrieval chunks produced by the preprocessing pipeline and can later be compared with the number of generated embeddings and indexed Qdrant points.

In [23]:
total_articles = (
    df_results["articles"]
    .fillna(0)
    .sum()
)

total_chunks = (
    df_results["chunks"]
    .fillna(0)
    .sum()
)

print(f"Total de artigos: {int(total_articles)}")
print(f"Total de chunks: {int(total_chunks)}")


Total de artigos: 1772
Total de chunks: 2556


## 2. Project Paths

The notebook operates on the parsed legal documents stored under `data/processed/`.

Paths are defined relative to the project root so that the chunking pipeline remains portable and does not depend on machine-specific absolute locations.

In [24]:
# Validar unicidade dos chunk_id em todo o corpus

all_chunk_ids = []

for document_id in df_results["document_id"]:

    chunks_path = (
        PROCESSED_DIR
        / document_id
        / "chunks.json"
    )

    if not chunks_path.exists():
        continue

    with open(
        chunks_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    all_chunk_ids.extend(
        chunk["chunk_id"]
        for chunk in data["chunks"]
    )

print(f"Chunks: {len(all_chunk_ids)}")
print(f"IDs únicos: {len(set(all_chunk_ids))}")

assert (
    len(all_chunk_ids)
    ==
    len(set(all_chunk_ids))
), "Existem chunk_id duplicados."

print("Todos os chunk_id são únicos.")


Chunks: 2556
IDs únicos: 2556
Todos os chunk_id são únicos.


## 2. Project Paths

The notebook operates on the parsed legal documents stored under `data/processed/`.

Paths are defined relative to the project root so that the chunking pipeline remains portable and does not depend on machine-specific absolute locations.

In [25]:
# Verificar chunks vazios e chunks acima do limite esperado

validation_rows = []

for document_id in df_results["document_id"]:

    chunks_path = (
        PROCESSED_DIR
        / document_id
        / "chunks.json"
    )

    if not chunks_path.exists():
        continue

    with open(
        chunks_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    for chunk in data["chunks"]:
        validation_rows.append({
            "document_id": chunk["document_id"],
            "chunk_id": chunk["chunk_id"],
            "article": chunk["article"],
            "word_count": chunk["word_count"],
            "empty_text": not bool(chunk["text"].strip())
        })

df_validation = pd.DataFrame(validation_rows)

print(
    "Chunks vazios:",
    int(df_validation["empty_text"].sum())
)

print(
    "Maior chunk:",
    int(df_validation["word_count"].max())
)

df_validation.sort_values(
    "word_count",
    ascending=False
).head(10)


Chunks vazios: 0
Maior chunk: 350


,document_id,chunk_id,article,word_count,empty_text
1243,CIVA,CIVA_ART_0144_CHUNK_001,Artigo 29.º,350,False
1182,CIVA,CIVA_ART_0123_CHUNK_002,Artigo 7.º,350,False
618,CIRS,CIRS_ART_0033_CHUNK_002,Artigo 12.º-A,350,False
1771,LGT,LGT_ART_0084_CHUNK_004,Artigo 68.º,350,False
74,CIMI,CIMI_ART_0063_CHUNK_001,Artigo 25.º,350,False
1568,EBF,EBF_ART_0125_CHUNK_002,Artigo 44.º-B,350,False
97,CIMI,CIMI_ART_0081_CHUNK_002,Artigo 43.º,350,False
2362,RGIT,RGIT_ART_0124_CHUNK_001,Artigo 119.º,350,False
2363,RGIT,RGIT_ART_0124_CHUNK_002,Artigo 119.º,350,False
1836,OE2026,OE2026_ART_0008_CHUNK_005,Artigo 8.º,350,False


## 2. Project Paths

The notebook operates on the parsed legal documents stored under `data/processed/`.

Paths are defined relative to the project root so that the chunking pipeline remains portable and does not depend on machine-specific absolute locations.

In [26]:
# Amostra final de chunks para inspeção manual

sample_columns = [
    "document_id",
    "chunk_id",
    "article",
    "article_title",
    "page_start",
    "page_end",
    "word_count",
    "text"
]

all_chunks_sample = []

for document_id in df_results["document_id"]:

    chunks_path = (
        PROCESSED_DIR
        / document_id
        / "chunks.json"
    )

    if not chunks_path.exists():
        continue

    with open(
        chunks_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    all_chunks_sample.extend(
        data["chunks"][:3]
    )

pd.DataFrame(
    all_chunks_sample
)[sample_columns]


,document_id,chunk_id,article,article_title,page_start,page_end,word_count,text
0,CIMI,CIMI_ART_0001_CHUNK_001,Artigo 13.º,Elementos para avaliações,8,8,113,1 - Os elementos referidos nas alíneas a) a d)...
1,CIMI,CIMI_ART_0002_CHUNK_001,Artigo 14.º,Taxas de conservação de esgotos,8,8,95,"1 - As taxas de conservação de esgotos, calcul..."
2,CIMI,CIMI_ART_0003_CHUNK_001,Artigo 15.º,Avaliação de prédios já inscritos na matriz,8,9,348,"1 - Revogado. (Lei nº 60-A/2011, de 30 de nove..."
3,CIRC,CIRC_ART_0001_CHUNK_001,Artigo 1.º,Pressuposto do imposto,17,17,32,O imposto sobre o rendimento das pessoas colet...
4,CIRC,CIRC_ART_0002_CHUNK_001,Artigo 2.º,Sujeitos passivos,17,17,176,1 - São sujeitos passivos do IRC: a) As socied...
5,CIRC,CIRC_ART_0003_CHUNK_001,Artigo 3.º,Base do imposto,17,18,337,1 - O IRC incide sobre: a) O lucro das socieda...
6,CIRS,CIRS_ART_0001_CHUNK_001,Artigo 1.º,Aprovação do Código,7,7,19,É aprovado o Código do Imposto sobre o Rendime...
7,CIRS,CIRS_ART_0002_CHUNK_001,Artigo 2.º,Entrada em vigor,7,7,13,O Código do IRS entra em vigor em 1 de janeiro...
8,CIRS,CIRS_ART_0003_CHUNK_001,Artigo 3.º,Impostos abolidos,7,7,98,1 - Na data da entrada em vigor do Código são ...
9,CIS,CIS_ART_0001_CHUNK_001,Artigo 13.º,Elementos para avaliações,5,5,113,1 - Os elementos referidos nas alíneas a) a d)...


## Next Step

The chunk files generated by this notebook are the input to:

```text
04_embedding_generation.ipynb
```

The next notebook converts each legal chunk into a semantic vector representation using **BGE-M3**. The resulting embeddings preserve the chunk metadata and are subsequently indexed in Qdrant for semantic retrieval.
